# Single GPU Training on CIFAR-10

In this notebook we train a CNN on the CIFAR-10 image classification dataset using a single GPU. CIFAR-10 has 60,000 32x32 color images across 10 classes.

By the end of this notebook you'll have:
- A trained model saved to `cifar_cnn.pth`
- Per-epoch training and test accuracy
- An inference demo on a random test image

## Imports and Constants

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import random
import time

BATCH_SIZE = 256
EPOCHS = 100
LEARNING_RATE = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
MODEL_WEIGHTS_PATH = 'cifar_cnn.pth'
CIFAR_MEAN = (0.49139968, 0.48215827, 0.44653124)
CIFAR_STD  = (0.24703233, 0.24348505, 0.26158768)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

## Device Detection

PyTorch can run on different hardware without changing any model code. We check what's available and use the best option — NVIDIA GPU (CUDA), Apple GPU (MPS), or CPU as a fallback.

In [ ]:
def get_device():
    # pytorch allows use to easily use different hardware without changing your model code
    # it has built in methods to detect whethere we are on a cude machine (gpu) or a mac with an apple gpu (mps)
    # if none of those exist running on a cpu will always work
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')

device = get_device()
print('Using device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Data Loading and Augmentation

CIFAR-10 comes with 50,000 training images and 10,000 test images. We apply random augmentations to the training set — horizontal flips and random crops — to help the model learn features that aren't tied to exact positions in the image. The test set just gets normalized, no augmentations.

In [ ]:
def get_cifar_data_transforms():
    # create a set of augmentations we want to do to our training and data set.
    # for the training set we randomly flip the image to better generalize
    # and also randomly crop out a small portion of it
    # then we normalize the rgb values to be ~N(0,1)
    # the values for normalization can be derived from the dataset itself
    # or found here https://stackoverflow.com/questions/66678052/how-to-calculate-the-mean-and-the-std-of-cifar10-data
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])

    # We do not augment the data if testing our classifier because we do not want to hinder the model in any way
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])

    return transform_train, transform_test


def get_cifar_data_sets(transform_train, transform_test):
    # pytorch has some datasets built into it that make it extremely easy to download an use
    # this is the CIFAR10 Datasets which contains images for 10 different classes that we can then classify
    # we can decide whether to download the training or testing set and keep them seperate
    # we can also choose to download the datasets to disk to make loading them faster in the future
    # finally we can apply the transforms we created earlier to the data as it comes in
    train_set = datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train,
    )
    test_set = datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test,
    )
    return train_set, test_set


def cifar10_loaders(batch_size=BATCH_SIZE):
    transform_train, transform_test = get_cifar_data_transforms()
    train_set, test_set = get_cifar_data_sets(transform_train, transform_test)

    # dataloaders make it easy to iterate over all of the training data
    # batch size specifies how many data samples to load for each call of the loader
    # shuffle makes sure we get the samples in a different random order for each epoch
    # num workers creates sub processes to help load the data faster
    # that is useful because if we are training on a gpu, we can set aside a number of threads dedicated to transfering data over
    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4
    )
    test_loader = DataLoader(
        test_set, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4
    )
    return train_loader, test_loader


train_loader, test_loader = cifar10_loaders()
print(f'Training batches: {len(train_loader)}')
print(f'Test batches:     {len(test_loader)}')

## Model Architecture: CIFARNet

Our CNN has 4 convolutional layers followed by 3 fully connected layers.

Conv layers learn spatial features like edges and textures. As we go deeper, we double the number of feature maps while halving the image dimensions with max pooling. At the end we flatten everything and pass it through the FC layers to get a 10-class prediction.

```
(B, 3, 32, 32)  ->  Conv+Pool+BN+ReLU  ->  (B, 64, 16, 16)
                ->  Conv+Pool+BN+ReLU  ->  (B, 128, 8, 8)
                ->  Conv+BN+ReLU       ->  (B, 256, 8, 8)
                ->  Conv+BN+ReLU       ->  (B, 512, 8, 8)
                ->  Flatten            ->  (B, 32768)
                ->  FC+ReLU+Dropout    ->  (B, 512)
                ->  FC+ReLU+Dropout    ->  (B, 256)
                ->  FC                 ->  (B, 10)
```

In [ ]:
# all networks must extend the nn.Module
class CIFARNet(nn.Module):
    def __init__(self):
        super().__init__()
        # networks must define a constructor to create all of the layers that the network will use

        # our network has 4 convolution layers with 3 mlp layers
        # Our original images have a size of 3x32x32 (3 for rgb then 32x32 image sizes)
        # so we can view each image as having 3 channels i.e 3 seperate images
        # and we convolute over that with 64 different kernels to create 64 new images
        # the convulution matrix will look over all input channels at once so it will take all rgb into account in one go
        # we then create several more convolution layers increasing the number of channels each time
        # kernel_size 3 means we convolute our images with a 3x3 matrix, padding 1 means we make our image 1 pixel bigger on each side
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)

        # max pool is an operation that just takes the max from a 2x2 grid
        # it is used to reduce the size of images for faster processing without missing key details
        self.pool = nn.MaxPool2d(2, 2)

        # Batch norm normalizes an entire input to the layer of a neural network to keep all activations reasonable
        # we batch normalize for each convolution layer
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(256)
        self.bn4 = nn.BatchNorm2d(512)

        # dropout means to randomly turn off half the nodes to prevent overfitting
        # this is only used during the training phase and not during testing
        self.dropout = nn.Dropout(p=0.5)

        # the final step in our network is the full connected layers of which we have 3 of
        # We have as input to the first linear layer 512 * 8 * 8 nodes since the last layer of convolution
        # has dimention (B, 512, 8, 8) so that allows one node for each pixel/channel
        # the last layer has 10 nodes since there are 10 different classes we want to identify
        self.fc1 = nn.Linear(512 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        # the forward method defines how data is passed through the layers we created in init

        # for our convolution layers we follow the same basic pattern
        # convlute the input tensor (the raw image)
        # max pool it to reduce the size
        # batch normalize
        # relu (activation function) max(0, x)
        # starting data size is (B, 3, 32, 32)
        x = self.conv1(x) # (B, 64, 32, 32) is the output where B is batch size
        x = self.pool(x)  # (B, 64, 16, 16) output from pool, notice the size of our data is cut in 1/4
        x = self.bn1(x)   # batch normalization doesn't change the size of the data just normalizes the activations
        x = F.relu(x)     # (B, 64, 16, 16) doesn't change the size of the data

        x = self.conv2(x) # (B, 128, 16, 16)
        x = self.pool(x)  # (B, 128, 8, 8) notice size of data cut in 1/4 again
        x = self.bn2(x)
        x = F.relu(x)     # (B, 128, 8, 8)

        x = self.conv3(x) # (B, 256, 8, 8)
        # we don't pool here since our image is already now pretty small
        x = self.bn3(x)
        x = F.relu(x)

        x = self.conv4(x) # (B, 512, 8, 8)
        x = self.bn4(x)
        x = F.relu(x)

        # final shape of data after all convolutions (B, 512, 8, 8)

        # to prepare to put our tensore in the neural network we must flatten it
        # the 1 means to start flattening with the first dimension so only the 512, 8, 8 are flattened
        # this is critical because without that we would flatten all batches into one giant tensore which would be useless
        x = torch.flatten(x, 1) # (B, 512 * 8 * 8) = (B, 32768) output of flatten

        # now we can pass through our flattened tensore into the fully connected layer and another relu activation
        x = self.fc1(x) # (B, 512)
        x = F.relu(x)

        # randomly turn some of the values to 0 to prevent overfitting
        x = self.dropout(x)
        x = self.fc2(x) # (B, 256)
        x = F.relu(x)

        x = self.dropout(x)
        # pass through last fc layer but do not relu because we will softmax later to get probabilities for each class
        x = self.fc3(x) # (B, 10) final size of output

        return x

## Training

We use SGD with momentum and a cosine annealing learning rate schedule. Cosine annealing starts with a high LR for fast initial learning and gradually decays it to zero over 100 epochs for fine-grained adjustments near the end.

In [ ]:
model = CIFARNet().to(device)
# Cross entropy loss is typically used for classification since it effectively computes the difference between 2 probabilities
loss_function = nn.CrossEntropyLoss()
# the optimizer is how we take our steps to adjust the weights
# SGD stands for stochastic gradient descent and it keeps track of momentum to avoid local minima
# but overall generally follows the negative gradient downwards
optimizer = optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)
# automatically changes the learning rate to learn faster at the beginning and then more precise later
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch + 1}/{EPOCHS}')
    start_time = time.time()

    # TRAIN LOOP

    # .train puts our model in training mode which activates features such as dropout
    model.train()
    train_correct = 0
    train_total = 0

    # pull out the images and labels for one batch
    for images, labels in train_loader:
        # we have to move everything to the device we are using
        images, labels = images.to(device), labels.to(device)

        # zeros out all of the stored gradients in our model so they don't accumlate across runs
        optimizer.zero_grad()

        # run the images through the model to obtain our output
        # output is like a set of probabilities for every class
        # we can take our prediction to be the class with the highest probability
        # this calls the forward method of our model that we implemented
        outputs = model(images)
        # loss measures how far our actual probabilities are from the one hot encoded truth labels
        # one hot encoded means the actual labels have a 0 for every class except the correct class
        # for one image the outputs could look something like this [0.1, 0.2, 0.7]
        # with the truth labels looking something like this [0, 0, 1]
        # and then loss would be how far apart those tensors are
        loss = loss_function(outputs, labels)
        # computes the gradients for every parameter in the model wrt to the loss we calculated
        loss.backward()
        # nudges the parameters by the gradient we just calculated to make the model actually learn
        optimizer.step()

        # track training accuracy
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    # update our optimizers learning rate
    scheduler.step()

    train_accuracy = 100 * train_correct / train_total
    print(f'Training Accuracy: {train_accuracy:.2f}%')

    # TEST LOOP
    # eval turns off things that help with training but hinder performance like dropout
    model.eval()
    correct, total = 0, 0

    # disables gradient calculation so our model can't learn from the testing data and cheat
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    accuracy = 100 * correct / total
    print(f'Test Accuracy: {accuracy:.2f}%')
    print(f'Epoch time: {(time.time() - start_time):.2f}s')

# save our weights so we can later use them for inference without having to retrain
torch.save(model.state_dict(), MODEL_WEIGHTS_PATH)
print(f'\nSaved model weights to {MODEL_WEIGHTS_PATH}')

## Inference

Load the saved weights and test on a random image from the test set. The image gets un-normalized before display so the colors look right — the model still received the normalized version.

In [ ]:
# load the trained model and put it on the device we are using
model_inf = CIFARNet().to(device)
model_inf.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
model_inf.eval()

# get our test data set
transform_train, transform_test = get_cifar_data_transforms()
_, test_set = get_cifar_data_sets(transform_train, transform_test)

# pick a random test image
idx = random.randint(0, len(test_set) - 1)
image, label = test_set[idx]

# save unnormalized copy for display — reverse the normalization so colors look right
unnorm = image.clone()
for t, m, s in zip(unnorm, CIFAR_MEAN, CIFAR_STD):
    t.mul_(s).add_(m)
np_img = unnorm.permute(1, 2, 0).numpy()

# run inference
image_batch = image.unsqueeze(0).to(device)
with torch.no_grad():
    output = model_inf(image_batch)
    predicted_idx = output.argmax(dim=1).item()

predicted_class = CLASSES[predicted_idx]
true_class = CLASSES[label]

print('------- Inference Result -------')
print(f'Random Test Index: {idx}')
print(f'Predicted class:   {predicted_class}')
print(f'True label:        {true_class}')
print('--------------------------------')

plt.figure(figsize=(4, 4))
plt.imshow(np_img)
plt.title(f'Predicted: {predicted_class}\nActual: {true_class}', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()